# B 四相位盲时间对应与分段归位：卡58用户运行包

卡57已采用，卡58实现与合成CPU核验完成，卡59发布到现有dev分支。真实GPU任务由人类运行，当前没有新真实结果。
Notebook与源码ZIP同步发布；直接Run all通过默认普通URL下载源码，无需手传ZIP；也可使用实际可用的源码目录/ZIP/URL。
固定Run all：加载同次P/S保存RGB8，crop[37:126]与delete[45:88]；四相位真实E，原22 pair pilot，有限单缺口族，完整最大集合与共识单位归位。
净32位，VAEload1/E16/FFT16；D/codec/新媒体解码/生成/梯度/路径后E均0。写端与原clean35保持不变。
不是唯一全帧/接缝恢复保证，ORACLE不是准确率必然上界。保留全部失败、结构缺失、错归属、false coverage与并列。
attempt先持久化，失败/中断耗掉这次尝试，之后Run all只回读；无重试/继续运行模式。
源码可编辑，不用hash/manifest/Git状态/精确版本作门禁。当前CPU证据不能代替实际变长VAE、GPU耗时、显存或科学结论。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-Blind-Temporal-Segmented-V1/fixed-four-phase-v1')
def fixed_denominator():
    return dict(receiver_rows=8, paired_windows=852, roi_scalars=74976, pooled_scalars=18744,
        received_payload_units=884, signed_votes=848640, zero_masks=848640, payload_tables=24,
        planned_bits=768, structurally_present_bits=576, structurally_missing_bits=192,
        per_mode=dict(planned_bits=256, structurally_present_bits=192, structurally_missing_bits=64,
                      source_states=176, source_cells=352, received_frames=908))

def execution_artifacts(output):
    output = Path(output)
    names = ('attempt.json', 'result.json', 'checkpoint.json', 'ledger.json', 'protocol.json',
             'plans_completion.json', 'blind_completion.json', 'evaluation.json', 'evaluation_truth.json')
    found = [n+s for n in names for s in ('', '.tmp') if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for folder in ('blind', 'features', 'observations', 'post_blind', 'prepared'):
        root = output/folder
        if root.is_dir():
            found.extend(str(p.relative_to(output)) for p in root.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)

def placeholder_evaluations(status='NOT_ENTERED'):
    """Stdlib-only fixed display skeleton, also embedded in the notebook."""
    out = {}
    for arm in ('P', 'S'):
        for view, n in (('CROP_MID', 89), ('DELETE43_PHASE', 138)):
            for key in ('CORRECT', 'WRONG'):
                for mode in ('EST', 'UNALIGNED', 'ORACLE'):
                    out[f'{arm}/{view}/{key}/{mode}'] = dict(status=status,
                        frames=[dict(received=i, status=status, exact=None, absolute_error=None) for i in range(n)],
                        source_cells=[dict(source_t=t, status=status) for t in range(1, 45)],
                        source_states=[dict(state=k, status=status) for k in range(22)],
                        bits=[dict(bit=i, status=status, decoded=None, correct=None, structurally_present=None,
                                   false_coverage=None) for i in range(32)],
                        denominators=dict(frames=n, cells=44, states=22, bits=32, structurally_present=24, structurally_missing=8))
    return out

def read_saved_attempt(output, artifacts):
    """Read only; missing rows remain unavailable without truth or new inference."""
    output = Path(output)
    saved, errors, name_used = {}, {}, None
    for name in ('result.json', 'checkpoint.json', 'result.json.tmp', 'checkpoint.json.tmp'):
        if not (output/name).is_file():
            continue
        try:
            candidate = json.loads((output/name).read_text())
            if not isinstance(candidate, dict):
                raise ValueError('saved record is not an object')
            saved, name_used = candidate, name
            break
        except (OSError, ValueError) as exc:
            errors[name] = f'{type(exc).__name__}: {exc}'
    tables = placeholder_evaluations('UNAVAILABLE_SAVED_ATTEMPT')
    stored_tables = saved.get('evaluation', {})
    if not isinstance(stored_tables, dict):
        errors['evaluation'] = 'invalid saved evaluation object; fixed unavailable rows displayed'
        stored_tables = {}
    for slot, table in stored_tables.items():
        if slot not in tables:
            continue
        expected = tables[slot]
        if isinstance(table, dict) and all(isinstance(table.get(field), list) and len(table[field]) == len(expected[field])
                for field in ('bits', 'frames', 'source_cells', 'source_states')):
            tables[slot] = table
        else:
            errors['evaluation/'+slot] = 'incomplete saved table; fixed unavailable rows displayed'
    calls, counts = None, saved.get('counts')
    ledger_path = output/'ledger.json'
    if ledger_path.is_file():
        try:
            calls = json.loads(ledger_path.read_text())
            if not isinstance(calls, dict) or not all(isinstance(row, dict) and 'kind' in row and 'status' in row for row in calls.values()):
                raise ValueError('invalid saved ledger rows')
            counts = {}
            for row in calls.values():
                group = counts.setdefault(row['kind'], dict(budget=0, started=0, returned=0, failed=0, interrupted=0, active=0))
                group['budget'] += 1
                group['started'] += row['status'] in ('STARTED', 'RETURNED', 'FAILED', 'INTERRUPTED')
                group['returned'] += row['status'] == 'RETURNED'
                group['failed'] += row['status'] == 'FAILED'
                group['interrupted'] += row['status'] == 'INTERRUPTED'
                group['active'] += row['status'] == 'STARTED'
        except (OSError, ValueError, TypeError) as exc:
            errors['ledger.json'] = f'{type(exc).__name__}: {exc}'
            calls, counts = None, saved.get('counts')
    return dict(status='EXISTING_ATTEMPT_NOT_REEXECUTED' if name_used else 'EXISTING_ATTEMPT_WITHOUT_CHECKPOINT',
        saved=saved, saved_record=name_used, existing_artifacts=artifacts, read_errors=errors,
        evaluation=tables, fixed_denominator=fixed_denominator(), counts=counts, calls=calls,
        count_source='ledger.json' if calls is not None else name_used,
        interpretation='Saved STARTED/unavailable rows stay spent; no model, FFT, inference or truth read')

EXISTING_ARTIFACTS = execution_artifacts(OUTPUT)
if EXISTING_ARTIFACTS:
    print('Existing attempt: saved records only', EXISTING_ARTIFACTS[:12])
else:
    import stat, urllib.request, zipfile
    # Published with the notebook on the existing dev branch; ordinary editable source.
    SOURCE_LOCATION = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/dev/local-joint-state-payload-v1/notebooks/blind_temporal_segmented_v1_portable_source.zip'
    source = Path(SOURCE_LOCATION)
    SOURCE_ROOT = Path('/content/b_blind_temporal_source')
    runner = 'experiments/wan_state_clock/blind_temporal_segmented_v1_run.py'
    if source.is_dir():
        SOURCE_ROOT = source
    elif not (SOURCE_ROOT/runner).is_file():
        archive = source
        if SOURCE_LOCATION.startswith(('https://','http://')):
            archive = Path('/content/b_blind_temporal_source.zip')
            if not archive.is_file():
                partial = archive.with_suffix('.download')
                urllib.request.urlretrieve(SOURCE_LOCATION, partial)
                partial.replace(archive)
        SOURCE_ROOT.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(archive) as bundle:
            for info in bundle.infolist():
                target = (SOURCE_ROOT/info.filename).resolve()
                if not target.is_relative_to(SOURCE_ROOT.resolve()) or stat.S_ISLNK(info.external_attr>>16):
                    raise ValueError('unsafe source archive path')
            bundle.extractall(SOURCE_ROOT)
    if not (SOURCE_ROOT/runner).is_file():
        raise FileNotFoundError('actual blind temporal runner missing')
    OUTPUT.mkdir(parents=True,exist_ok=True)
    print('Using available source',SOURCE_ROOT)


依赖仅修复实际缺包/已知导入冲突，复用已有Wan VAE缓存。已有attempt时不下载源码、不安装、不模型调用。真实16E/FFT仍由人类未来运行；这里没有隐藏测试模式。

In [ ]:

import subprocess, sys

EXISTING_ARTIFACTS = execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/"setup.log"
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode

    # Copied from the already-used A/P2 setup; no A import/build dependency.
    def ensure_imports(probe, packages, label):
        import re
        module_packages = {package.replace('-', '_'): package for package in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], ('Recheck ' if repaired else 'Probe ')+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named ['\"]([^'\"]+)['\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1) in module_packages:
                package = module_packages[missing.group(1)]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' failed without a known targeted repair; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, 'Repair '+label+': '+action)

    ensure_imports('import numpy,torch', ['numpy', 'torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub; from diffusers import AutoencoderKLWan',
                   ['diffusers', 'accelerate', 'safetensors', 'huggingface-hub'], 'Wan VAE imports')
    with (OUTPUT/"runner.log").open("w") as log:
        process = subprocess.Popen([sys.executable, "-u", "-m",
            "experiments.wan_state_clock.blind_temporal_segmented_v1_run"],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT)
    import time
    try:
        while process.poll() is None:
            time.sleep(30)
            ledger_path = OUTPUT/"ledger.json"
            if ledger_path.exists():
                ledger = json.loads(ledger_path.read_text())
                returned = sum(row["status"] == "RETURNED" for row in ledger.values())
                active = [name for name, row in ledger.items() if row["status"] == "STARTED"]
                print(f"Completed core units: {returned}/{len(ledger)}; active: {active}")
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print("Stopped; saved STARTED calls remain unresolved and will not be replayed.")
        raise
    print("Runner return code:", process.returncode)
else:
    print("Existing attempt: displaying saved records; no physical calls repeated.")
print("Output:", OUTPUT)


读取全部固定24表；规范BER/有效覆盖只评价结构存在位，结构缺失输出另列false coverage。完整最大集合、原始票、source cell/state/frame记录保存在输出目录。

In [ ]:

r = read_saved_attempt(OUTPUT, execution_artifacts(OUTPUT))
print('Readback:', r['status'], 'saved:', r['saved_record'])
print('Fixed denominator:', r['fixed_denominator'])
print('Call counts:', r['counts'])
print('Saved status:', r['saved'].get('status'))
print('Path statuses / full maximum counts:', r['saved'].get('paths'))
for name, table in r['evaluation'].items():
    print(name, table.get('summary', {'status':table['status'], 'planned_bits':len(table['bits'])}))
print('Common-bit BER / valid added and lost coverage / false coverage:', r['saved'].get('comparisons'))
for name in ('fatal_error','evaluation_error'):
    if r['saved'].get(name): print(name, r['saved'][name])
if r['read_errors']: print('Unreadable saved records:', r['read_errors'])
ledger_path = OUTPUT/'ledger.json'
if ledger_path.is_file():
    try:
        ledger = json.loads(ledger_path.read_text())
        for name, row in ledger.items():
            if row.get('status') != 'RETURNED': print('Retained slot',name,row)
    except (OSError,ValueError) as exc: print('Unreadable ledger',type(exc).__name__,str(exc))
print('No scientific PASS. Full fixed rows, ties and raw votes remain in:',OUTPUT)
